# Mindbaton brain 2B: make the model file (GGUF) from the trained layers

The training notebook (`brain-2b.ipynb`) uploaded the trained layers (`lora/`, 44 MB) to your Hugging Face repo. This
notebook turns them into the file Mindbaton downloads: it merges them into Qwen3.5-2B, converts the result with the same
llama.cpp build Mindbaton runs, **tests it with the exact request Mindbaton sends**, and uploads
`mindbaton-brain-2b.Q4_K_M.gguf` (~1.3 GB).

**No GPU needed.** Session options: **Accelerator: None**, **Internet: on**, and the secret **`HF_TOKEN`** (Write) ticked.
Then **Run All** — about 20 minutes.

In [ ]:
REPO = "1972521116s/mindbaton-brain-2b"   # your model on Hugging Face (lora/ is already there)
BASE = "Qwen/Qwen3.5-2B"
BUILD = "b11223"                          # the llama.cpp build Mindbaton runs
W = "/kaggle/working"
!git clone -q --depth 1 --branch {BUILD} https://github.com/ggml-org/llama.cpp {W}/llama.cpp
%pip install -q {W}/llama.cpp/gguf-py peft
!cd {W} && curl -sL -o q.tgz https://github.com/ggml-org/llama.cpp/releases/download/{BUILD}/llama-{BUILD}-bin-ubuntu-x64.tar.gz && tar xzf q.tgz && rm q.tgz
!git clone -q --depth 1 -b brain-upgrade https://github.com/DkshByte/mindbaton {W}/mindbaton

In [ ]:
# 1. Qwen3.5-2B + the trained layers, merged into one model (on the CPU: ~5 GB of memory)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
model = PeftModel.from_pretrained(base, REPO, subfolder="lora").merge_and_unload()
tokenizer = AutoTokenizer.from_pretrained(REPO, subfolder="lora")
model.save_pretrained(f"{W}/merged", safe_serialization=True)
tokenizer.save_pretrained(f"{W}/merged")
del model, base

In [ ]:
# 2. Convert and shrink to 4 bits with the llama.cpp build Mindbaton uses
!python {W}/llama.cpp/convert_hf_to_gguf.py {W}/merged --outtype f16 --outfile {W}/model-f16.gguf
!{W}/llama-{BUILD}/llama-quantize {W}/model-f16.gguf {W}/mindbaton-brain-2b.Q4_K_M.gguf Q4_K_M
!ls -la {W}/*.gguf

In [ ]:
# 3. Test it exactly as Mindbaton will: llama-server, the same request, the rules' check
import json, subprocess, sys, time, urllib.request
sys.path.insert(0, f"{W}/mindbaton")
import os; os.environ["MINDBATON_DATA"] = "/tmp/mb"
import deep
srv = subprocess.Popen([f"{W}/llama-{BUILD}/llama-server", "-m", f"{W}/mindbaton-brain-2b.Q4_K_M.gguf", "--port", "8099",
                        "-c", "2048", "-np", "1", "--jinja"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8099/health", timeout=2); break
    except Exception:
        time.sleep(1)
tests = {"5-a-side football every thursday after work": "plays", "zero caffeine since march": "avoids",
         "my dad plays tennis every sunday": "plays", "what's the best laptop for coding?": None}
ok = 0
for t, want in tests.items():
    t0 = time.time()
    got = deep.check(t, deep.ask("http://127.0.0.1:8099", t))
    good = (not got) if want is None else any(f["rel"] == want for f in got)
    ok += good
    print(f"{'ok ' if good else 'BAD'} {time.time() - t0:4.1f}s  {t} -> {got}")
srv.terminate()
assert ok >= 3, "the converted model doesn't answer like the trained one: don't upload it; paste this output to get it fixed"

In [ ]:
# 4. Upload the file Mindbaton downloads
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
api.upload_file(path_or_fileobj=f"{W}/mindbaton-brain-2b.Q4_K_M.gguf", path_in_repo="mindbaton-brain-2b.Q4_K_M.gguf", repo_id=REPO)
print("Done: https://huggingface.co/" + REPO + "/blob/main/mindbaton-brain-2b.Q4_K_M.gguf")